<a href="https://colab.research.google.com/github/prasansree/BusBookingApp/blob/main/c2_w1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Machine Learning: Unsupervised Learning

**Instructor:** Daniel Acuna, Ph.D.
**Position:** Associate Professor of Computer Science
**Institution:** University of Colorado Boulder

---

Lab 1: Introduction to Unsupervised Learning and Data Exploration

---

In this lab, you will explore the foundations of unsupervised learning by working
with the **USArrests dataset**. This dataset contains crime statistics for 50 US
states across 3 features: Murder, Assault, UrbanPop (urban population percentage).

You will:
- Perform exploratory data analysis (EDA) on unlabeled data
- Understand why feature scaling is critical for distance-based methods
- Implement distance metrics (Euclidean and Manhattan)
- Build distance matrices and find similar/dissimilar observations
- Apply standardization and observe its impact on distances

These concepts form the foundation for clustering and dimensionality reduction
techniques you will learn in upcoming weeks.

## Setup (do not edit)

In [3]:
import pathlib
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)

_DATA_PATH = pathlib.Path("sample_data/us_arrests.csv")
if not _DATA_PATH.exists():
    raise FileNotFoundError(
        "us_arrests.csv is missing from the lab directory. Please run the download "
        "script (w1_download_datasets.py) or ask the TA for assistance."
    )

## 1. Load the Dataset *(10 points)*

Load the USArrests dataset from `us_arrests.csv` into a pandas DataFrame.
The first column contains state names and should be used as the index.

Store the following in the required variables:
- **`q1_shape`**: A tuple containing the shape of the DataFrame `(n_rows, n_cols)`
- **`q1_columns`**: A list of the column names

In [9]:
# Grade Cell: Question 1
#
# Task: Load the USArrests dataset and explore its structure
#
# Instructions:
# 1. Read the CSV file using pd.read_csv() with index_col=0 to use state names as index
# 2. Store the shape as a tuple in q1_shape
# 3. Store the column names as a list in q1_columns

# your code here
df = pd.read_csv(_DATA_PATH, index_col=0)

q1_shape = df.shape

q1_columns = df.columns.tolist()

print("Dataset Shape:", q1_shape)
print("Column Names:", q1_columns)

df.describe()

Dataset Shape: (50, 3)
Column Names: ['Murder', 'Assault', 'UrbanPop']


,Murder,Assault,UrbanPop
count,50.00000,50.000000,50.000000
mean,7.78800,170.760000,65.540000
std,4.35551,83.337661,14.474763
min,0.80000,45.000000,32.000000
25%,4.07500,109.000000,54.500000
50%,7.25000,159.000000,66.000000
75%,11.25000,249.000000,77.750000
max,17.40000,337.000000,91.000000


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 1
assert isinstance(q1_shape, tuple), (
    "q1_shape must be a tuple. Use df.shape which returns (rows, cols)."
)
assert len(q1_shape) == 2, (
    "q1_shape should have 2 elements (rows, cols). "
    "Make sure you're using .shape, not .shape[0]."
)
assert q1_shape[0] > 0 and q1_shape[1] > 0, (
    "Shape values must be positive. Is your CSV loading correctly?"
)
assert isinstance(q1_columns, list), (
    "q1_columns must be a list. Use df.columns.tolist() to convert."
)
assert len(q1_columns) == q1_shape[1], (
    "Number of columns in q1_columns should match q1_shape[1]."
)
print(f"Dataset shape: {q1_shape}")
print(f"Columns: {q1_columns}")

Dataset shape: (50, 3)
Columns: ['Murder', 'Assault', 'UrbanPop']


## 2. Compute Summary Statistics *(10 points)*

Compute the **mean** and **standard deviation** for each feature in the dataset.

Store the results as dictionaries:
- **`q2_means`**: A dictionary mapping feature names to their mean values (rounded to 2 decimals)
- **`q2_stds`**: A dictionary mapping feature names to their standard deviation values (rounded to 2 decimals)

Use `ddof=0` for standard deviation (population std) to match sklearn's StandardScaler.

In [ ]:
# Grade Cell: Question 2
#
# Task: Compute summary statistics for each feature
#
# Instructions:
# 1. Calculate the mean of each column
# 2. Calculate the standard deviation of each column (use ddof=0)
# 3. Round values to 2 decimal places
# 4. Store as dictionaries with column names as keys

q2_means = df.mean().round(2).to_dict()
q2_stds = df.std(ddof=0).round(2).to_dict()
print(f"Feature Means: {q2_means}" )
print(f"Feature Standard Deviations:{q2_stds}")

Feature Means: {'Murder': 7.79, 'Assault': 170.76, 'UrbanPop': 65.54}
Feature Standard Deviations:{'Murder': 4.31, 'Assault': 82.5, 'UrbanPop': 14.33}


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 2
assert isinstance(q2_means, dict), (
    "q2_means must be a dictionary mapping feature names to mean values."
)
assert isinstance(q2_stds, dict), (
    "q2_stds must be a dictionary mapping feature names to std values."
)
assert len(q2_means) == 3, (
    "q2_means should have 3 entries, one for each feature."
)
assert len(q2_stds) == 3, (
    "q2_stds should have 3 entries, one for each feature."
)
assert all(isinstance(v, float) for v in q2_means.values()), (
    "All mean values should be floats. Use round() to ensure this."
)
assert all(v > 0 for v in q2_stds.values()), (
    "Standard deviations must be positive. Check your calculation."
)
print(f"Feature means: {q2_means}")
print(f"Feature stds: {q2_stds}")

Feature means: {'Murder': 7.79, 'Assault': 170.76, 'UrbanPop': 65.54}
Feature stds: {'Murder': 4.31, 'Assault': 82.5, 'UrbanPop': 14.33}


## 3. Understand Feature Scales *(10 points)*

One critical insight from unsupervised learning is that features with larger scales
can dominate distance calculations. To see this, compute the **range** (max - min)
for each feature.

Store the result as:
- **`q3_ranges`**: A dictionary mapping feature names to their range values (rounded to 2 decimals)

After computing, observe which feature has the largest range—this feature would
dominate Euclidean distances if we don't standardize!

In [10]:
# Grade Cell: Question 3
#
# Task: Compute the range (max - min) for each feature
#
# Instructions:
# 1. For each column, compute max() - min()
# 2. Round to 2 decimal places
# 3. Store as a dictionary

q3_ranges = (df.max() - df.min()).round(2).to_dict()

print(f"Feature Ranges: { q3_ranges}")


Feature Ranges: {'Murder': 16.6, 'Assault': 292.0, 'UrbanPop': 59.0}


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 3
assert isinstance(q3_ranges, dict), (
    "q3_ranges must be a dictionary mapping feature names to range values."
)
assert len(q3_ranges) == 3, (
    "q3_ranges should have 3 entries, one for each feature."
)
assert all(v > 0 for v in q3_ranges.values()), (
    "Range values must be positive. Check your max - min calculation."
)
# Check that Assault has the largest range (it should dominate distances)
max_range_feature = max(q3_ranges, key=q3_ranges.get)
print(f"Feature ranges: {q3_ranges}")
print(f"Feature with largest range: {max_range_feature}")

## 4. Implement Euclidean Distance *(10 points)*

Distance metrics are the foundation of clustering algorithms. Implement a function
that computes the **Euclidean distance** between two vectors.

The Euclidean distance formula is:
$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{j=1}^{p}(x_j - y_j)^2}$$

Create a function **`euclidean_distance(x, y)`** that:
- Takes two numpy arrays or pandas Series of the same length
- Returns the Euclidean distance as a float rounded to 3 decimal places

Then compute the distance between **Alabama** and **Alaska** (the first two states)
and store it in **`q4_distance_al_ak`**.

In [ ]:
# Grade Cell: Question 4
#
# Task: Implement Euclidean distance function
#
# Instructions:
# 1. Implement the euclidean_distance function using the formula above
# 2. Use np.sqrt() and np.sum() for the calculation
# 3. Round the result to 3 decimal places
# 4. Compute distance between Alabama and Alaska

def euclidean_distance(x, y):
    x = np.array(x)
    y = np.array(y)
    diff = x - y
    return round(np.sqrt(np.sum(diff ** 2)), 3)

alabama = df.iloc[0]
alaska = df.iloc[1]

q4_distance_al_ak = euclidean_distance(alabama, alaska)

print(q4_distance_al_ak)

28.97


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 4
assert callable(euclidean_distance), (
    "euclidean_distance should be a function. Did you define it with 'def'?"
)
# Test with known values
test_x = np.array([1, 2, 3])
test_y = np.array([4, 5, 6])
test_dist = euclidean_distance(test_x, test_y)
assert isinstance(test_dist, float), (
    "euclidean_distance should return a float."
)
assert test_dist >= 0, (
    "Distance must be non-negative. Check your formula."
)
assert isinstance(q4_distance_al_ak, float), (
    "q4_distance_al_ak must be a float."
)
assert q4_distance_al_ak > 0, (
    "Distance between different states should be positive."
)
print(f"Test distance [1,2,3] to [4,5,6]: {test_dist}")
print(f"Distance Alabama to Alaska: {q4_distance_al_ak}")

Test distance [1,2,3] to [4,5,6]: 5.196
Distance Alabama to Alaska: 28.97


## 5. Implement Manhattan Distance *(10 points)*

Manhattan distance is an alternative to Euclidean distance that is more robust to
outliers. It sums the absolute differences rather than squared differences.

The Manhattan distance formula is:
$$d(\mathbf{x}, \mathbf{y}) = \sum_{j=1}^{p}|x_j - y_j|$$

Create a function **`manhattan_distance(x, y)`** that:
- Takes two numpy arrays or pandas Series of the same length
- Returns the Manhattan distance as a float rounded to 3 decimal places

Then compute the Manhattan distance between **Alabama** and **Alaska** and
store it in **`q5_manhattan_al_ak`**.

In [ ]:
# Grade Cell: Question 5
#
# Task: Implement Manhattan distance function
#
# Instructions:
# 1. Implement the manhattan_distance function using the formula above
# 2. Use np.sum() and np.abs() for the calculation
# 3. Round the result to 3 decimal places
# 4. Compute distance between Alabama and Alaska

def manhattan_distance(x, y):
     x = np.array(x)
     y = np.array(y)
     return float(round(np.sum(np.abs(x - y)), 3))

alabama = df.iloc[0]
alaska = df.iloc[1]

q5_manhattan_al_ak = manhattan_distance(alabama, alaska)

print(q5_manhattan_al_ak)

40.2


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 5
assert callable(manhattan_distance), (
    "manhattan_distance should be a function. Did you define it with 'def'?"
)
# Test with known values
test_x = np.array([1, 2, 3])
test_y = np.array([4, 5, 6])
test_manhattan = manhattan_distance(test_x, test_y)
assert isinstance(test_manhattan, float), (
    "manhattan_distance should return a float."
)
assert test_manhattan >= 0, (
    "Distance must be non-negative. Check your formula."
)
# Manhattan should be >= Euclidean for the same points
assert test_manhattan >= euclidean_distance(test_x, test_y), (
    "Manhattan distance should be >= Euclidean distance for the same points."
)
assert isinstance(q5_manhattan_al_ak, float), (
    "q5_manhattan_al_ak must be a float."
)
print(f"Test Manhattan distance [1,2,3] to [4,5,6]: {test_manhattan}")
print(f"Manhattan distance Alabama to Alaska: {q5_manhattan_al_ak}")

Test Manhattan distance [1,2,3] to [4,5,6]: 9.0
Manhattan distance Alabama to Alaska: 40.2


## 6. Build a Distance Matrix *(15 points)*

A distance matrix shows the pairwise distances between all observations. This is
fundamental to many clustering algorithms.

Build a **Euclidean distance matrix** for all 50 states using the **unstandardized**
(original) data.

Store the result as:
- **`q6_dist_matrix`**: A 2D numpy array of shape (50, 50) containing pairwise distances

The diagonal should be 0 (distance from a state to itself), and the matrix should
be symmetric (distance from A to B equals distance from B to A).

**Hint:** You can use nested loops, or use `scipy.spatial.distance.pdist` and
`squareform` for efficiency.

In [ ]:
# Grade Cell: Question 6
#
# Task: Build a pairwise Euclidean distance matrix
#
# Instructions:
# 1. Create a 50x50 matrix of zeros
# 2. For each pair of states, compute Euclidean distance
# 3. Store distances in the matrix (it should be symmetric)

# Convert DataFrame to numpy array
from scipy.spatial.distance import pdist, squareform
X = df.values

# Compute pairwise Euclidean distances
q6_dist_matrix = squareform(pdist(X, metric='euclidean'))

print(q6_dist_matrix.shape)

(50, 50)


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Question 6
assert isinstance(q6_dist_matrix, np.ndarray), (
    "q6_dist_matrix must be a numpy array."
)
assert q6_dist_matrix.shape == (50, 50), (
    f"Distance matrix should be 50x50, got {q6_dist_matrix.shape}. "
    "Make sure you're computing distances for all pairs of states."
)
# Check diagonal is zeros (distance to self)
assert np.allclose(np.diag(q6_dist_matrix), 0), (
    "Diagonal elements should be 0 (distance from a state to itself)."
)
# Check symmetry
assert np.allclose(q6_dist_matrix, q6_dist_matrix.T), (
    "Distance matrix should be symmetric. d(A,B) = d(B,A)."
)
# Check all values are non-negative
assert (q6_dist_matrix >= 0).all(), (
    "All distances must be non-negative."
)
print(f"Distance matrix shape: {q6_dist_matrix.shape}")
print(f"Diagonal (should be zeros): {np.diag(q6_dist_matrix)[:5]}")
print(f"Max distance: {q6_dist_matrix.max():.2f}")

Distance matrix shape: (50, 50)
Diagonal (should be zeros): [0. 0. 0. 0. 0.]
Max distance: 293.57


## 7. Standardize the Data *(10 points)*

As we learned in the lectures, **standardization is critical** before computing
distances. Without it, features with larger scales dominate the distance calculation.

Apply z-score standardization to the data:
$$z_j = \frac{x_j - \mu_j}{\sigma_j}$$

Use `sklearn.preprocessing.StandardScaler` to standardize the data.

Store the results as:
- **`q7_scaled_data`**: A pandas DataFrame with the standardized values (same index and columns as original)
- **`q7_scaled_means`**: A dictionary of the means of the scaled data (should all be ~0)
- **`q7_scaled_stds`**: A dictionary of the stds of the scaled data (should all be ~1)

In [ ]:
# Grade Cell: Question 7
#
# Task: Standardize the data using StandardScaler
#
# Instructions:
# 1. Create a StandardScaler instance
# 2. Fit and transform the data
# 3. Convert back to DataFrame with original index and columns
# 4. Compute means and stds of the scaled data (should be ~0 and ~1)

# Initialize scaler
scaler = StandardScaler()

# Fit and transform data
scaled_array = scaler.fit_transform(df)

# Convert back to DataFrame (preserve index + columns)
q7_scaled_data = pd.DataFrame(scaled_array, index=df.index, columns=df.columns)

# Compute means and stds of scaled data
q7_scaled_means = q7_scaled_data.mean().round(2).to_dict()
q7_scaled_stds = q7_scaled_data.std(ddof=0).round(2).to_dict()

print("Scaled Means:", q7_scaled_means)
print("Scaled Stds:", q7_scaled_stds)

Scaled Means: {'Murder': -0.0, 'Assault': 0.0, 'UrbanPop': -0.0}
Scaled Stds: {'Murder': 1.0, 'Assault': 1.0, 'UrbanPop': 1.0}


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 7
assert isinstance(q7_scaled_data, pd.DataFrame), (
    "q7_scaled_data must be a pandas DataFrame."
)
assert q7_scaled_data.shape == df.shape, (
    "Scaled data should have same shape as original."
)
assert list(q7_scaled_data.columns) == list(df.columns), (
    "Scaled data should have same column names as original."
)
assert isinstance(q7_scaled_means, dict), (
    "q7_scaled_means must be a dictionary."
)
assert isinstance(q7_scaled_stds, dict), (
    "q7_scaled_stds must be a dictionary."
)
# Check that means are approximately 0
all_means_near_zero = all(abs(v) < 0.01 for v in q7_scaled_means.values())
assert all_means_near_zero, (
    "After standardization, all feature means should be ~0. "
    "Did you use StandardScaler correctly?"
)
# Check that stds are approximately 1
all_stds_near_one = all(abs(v - 1.0) < 0.01 for v in q7_scaled_stds.values())
assert all_stds_near_one, (
    "After standardization, all feature stds should be ~1. "
    "Did you use StandardScaler correctly?"
)
print(f"Scaled means (should be ~0): {q7_scaled_means}")
print(f"Scaled stds (should be ~1): {q7_scaled_stds}")

Scaled means (should be ~0): {'Murder': -0.0, 'Assault': 0.0, 'UrbanPop': -0.0}
Scaled stds (should be ~1): {'Murder': 1.0, 'Assault': 1.0, 'UrbanPop': 1.0}


## 8. Distance Matrix on Standardized Data *(10 points)*

Now build a distance matrix using the **standardized** data and compare it to
the unstandardized version.

Store the results as:
- **`q8_scaled_dist_matrix`**: A 2D numpy array of shape (50, 50) with pairwise distances on scaled data
- **`q8_al_ak_scaled`**: The Euclidean distance between Alabama and Alaska using scaled data (rounded to 3 decimals)

Compare `q8_al_ak_scaled` to `q4_distance_al_ak` to see how standardization changes distances.

In [ ]:
# Grade Cell: Question 8
#
# Task: Build distance matrix on standardized data
#
# Instructions:
# 1. Compute pairwise Euclidean distances on q7_scaled_data
# 2. Extract the distance between Alabama and Alaska

X_scaled = q7_scaled_data.values

# 1. Distance matrix on scaled data
q8_scaled_dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))

# 2. Alabama vs Alaska (first two states)
q8_al_ak_scaled = float(round(q8_scaled_dist_matrix[0, 1], 3))

print("Scaled distance (Alabama vs Alaska):", q8_al_ak_scaled)

Scaled distance (Alabama vs Alaska): 1.07


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 8
assert isinstance(q8_scaled_dist_matrix, np.ndarray), (
    "q8_scaled_dist_matrix must be a numpy array."
)
assert q8_scaled_dist_matrix.shape == (50, 50), (
    f"Distance matrix should be 50x50, got {q8_scaled_dist_matrix.shape}."
)
assert np.allclose(np.diag(q8_scaled_dist_matrix), 0), (
    "Diagonal elements should be 0."
)
assert np.allclose(q8_scaled_dist_matrix, q8_scaled_dist_matrix.T), (
    "Distance matrix should be symmetric."
)
assert isinstance(q8_al_ak_scaled, float), (
    "q8_al_ak_scaled must be a float."
)
print(f"Unstandardized Alabama-Alaska distance: {q4_distance_al_ak}")
print(f"Standardized Alabama-Alaska distance: {q8_al_ak_scaled}")
print(f"Ratio (unstandardized / standardized): {q4_distance_al_ak / q8_al_ak_scaled:.2f}")

Unstandardized Alabama-Alaska distance: 28.97
Standardized Alabama-Alaska distance: 1.07
Ratio (unstandardized / standardized): 27.07


## 9. Find the Most Similar States *(10 points)*

Using the **standardized distance matrix** from Question 8, find the pair of states
that are **most similar** (smallest non-zero distance).

Store the results as:
- **`q9_most_similar_pair`**: A tuple of two state names (strings) representing the closest pair
- **`q9_min_distance`**: The distance between them (rounded to 3 decimals)

**Hint:** Set diagonal to infinity before finding the minimum, since diagonal is 0.

In [ ]:
# Grade Cell: Question 9
#
# Task: Find the two most similar states based on standardized distances
#
# Instructions:
# 1. Copy the distance matrix and set diagonal to infinity
# 2. Find the indices of the minimum value
# 3. Get the state names corresponding to those indices

dist_matrix = q8_scaled_dist_matrix.copy()

# Step 1: ignore diagonal
np.fill_diagonal(dist_matrix, np.inf)

# Step 2: find minimum distance index
min_idx = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)

i, j = min_idx

# Step 3: extract state names
states = df.index.tolist()
q9_most_similar_pair = (states[i], states[j])

# Step 4: extract distance
q9_min_distance = float(round(dist_matrix[i, j], 3))

print("Most similar states:", q9_most_similar_pair)
print("Minimum distance:", q9_min_distance)

Most similar states: ('Iowa', 'New Hampshire')
Minimum distance: 0.075


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 9
assert isinstance(q9_most_similar_pair, tuple), (
    "q9_most_similar_pair must be a tuple of two state names."
)
assert len(q9_most_similar_pair) == 2, (
    "q9_most_similar_pair should contain exactly 2 state names."
)
assert all(isinstance(s, str) for s in q9_most_similar_pair), (
    "Both elements in q9_most_similar_pair should be strings (state names)."
)
assert q9_most_similar_pair[0] != q9_most_similar_pair[1], (
    "The two states should be different. Did you exclude the diagonal?"
)
assert isinstance(q9_min_distance, float), (
    "q9_min_distance must be a float."
)
assert q9_min_distance > 0, (
    "Minimum distance should be positive (not self-distance)."
)
print(f"Most similar states: {q9_most_similar_pair}")
print(f"Distance between them: {q9_min_distance}")

Most similar states: ('Iowa', 'New Hampshire')
Distance between them: 0.075


## 10. Find k-Nearest Neighbors *(5 points)*

For a given anchor state, find its **k nearest neighbors** based on standardized
Euclidean distance.

Implement a function **`find_k_nearest(state_name: str, k: int) -> List[str]`** that:
- Takes a state name and number of neighbors k
- Returns a list of the k nearest state names (excluding the anchor state itself)

Then find the **3 nearest neighbors of California** and store them in **`q10_ca_neighbors`**.

In [ ]:
# Grade Cell: Question 10
#
# Task: Implement k-nearest neighbors lookup
#
# Instructions:
# 1. Get the row of distances for the given state
# 2. Sort indices by distance (excluding the state itself)
# 3. Return the top k state names

# Precompute index lookup
states = df.index.tolist()

def find_k_nearest(state_name: str, k: int):
    # get index of the state
    idx = states.index(state_name)

    # get distances for that state
    distances = q8_scaled_dist_matrix[idx].copy()

    # exclude itself
    distances[idx] = np.inf

    # get indices of k smallest distances
    nearest_idx = np.argsort(distances)[:k]

    # return state names
    return [states[i] for i in nearest_idx]

# 3 nearest neighbors of California
q10_ca_neighbors = find_k_nearest("California", 3)

print(q10_ca_neighbors)

['New York', 'Illinois', 'Arizona']


In [ ]:
# If all tests pass (there might be hidden tests), you will earn 5 points
# Test Cell: Question 10
assert callable(find_k_nearest), (
    "find_k_nearest should be a function."
)
assert isinstance(q10_ca_neighbors, list), (
    "q10_ca_neighbors must be a list of state names."
)
assert len(q10_ca_neighbors) == 3, (
    "q10_ca_neighbors should contain exactly 3 neighbors."
)
assert all(isinstance(s, str) for s in q10_ca_neighbors), (
    "All neighbors should be strings (state names)."
)
assert "California" not in q10_ca_neighbors, (
    "California should not be in its own neighbor list."
)
# Test the function with another state
test_neighbors = find_k_nearest("Texas", 2)
assert len(test_neighbors) == 2, (
    "find_k_nearest(Texas, 2) should return 2 neighbors."
)
assert "Texas" not in test_neighbors, (
    "A state should not appear in its own neighbor list."
)
print(f"California's 3 nearest neighbors: {q10_ca_neighbors}")
print(f"Texas's 2 nearest neighbors: {test_neighbors}")

California's 3 nearest neighbors: ['New York', 'Illinois', 'Arizona']
Texas's 2 nearest neighbors: ['Nevada', 'Michigan']


## Next Steps

Congratulations on completing the assignment! Before submitting:

1. Make sure all your cells run without errors.
2. Ensure you've answered all parts of each question.
3. If any autograder tests fail, revisit your answers.

**Key Takeaways from This Lab:**

- **Unsupervised learning** works with unlabeled data to discover patterns
- **Feature scales matter**: Without standardization, large-scale features dominate distances
- **Distance metrics** (Euclidean, Manhattan) quantify similarity between observations
- **Standardization** (z-score) puts all features on equal footing
- **Distance matrices** are the foundation for clustering algorithms

In the next week, you'll use these concepts to perform dimensionality reduction
with Principal Component Analysis (PCA)!